# Multi-Agent Reliability Study
## Communication Protocol Efficiency & Failure Propagation

**GR5293 Final Project Demo — Self-contained Mock Version**

This notebook implements a 4-agent pipeline for analyzing Apple earnings reports:
- **Orchestrator**: Decomposes the task and coordinates agents
- **Retriever**: Fetches financial data
- **Analyst**: Interprets and analyzes data
- **Writer**: Generates the final report

Three communication protocols are tested:
- `NL`: Natural Language (plain text)
- `JSON`: Structured format
- `SHARED`: Shared memory (global state dict)

Four failure injection modes are tested:
- `NONE`: Normal operation
- `TOOL_FAIL`: Retriever returns empty result
- `CONTRADICTION`: Two data sources conflict
- `AMBIGUITY`: Task description is vague
- `RESOURCE_LIMIT`: Max tool calls = 1

> **Note**: This version uses mock LLM calls (rule-based stubs). Swap `MockLLM` for a real API client to run with DeepSeek/OpenAI.

## 0. Install Dependencies

In [1]:
# All standard library — no pip installs needed for mock version
import json
import time
import copy
import random
import hashlib
import textwrap
from enum import Enum
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports OK")

✅ All imports OK


## 1. Protocol & Failure Mode Definitions

In [2]:
class Protocol(str, Enum):
    NL     = "NL"      # Natural Language
    JSON   = "JSON"    # Structured JSON
    SHARED = "SHARED"  # Shared memory state

class FailureMode(str, Enum):
    NONE           = "NONE"           # Baseline
    TOOL_FAIL      = "TOOL_FAIL"      # Retriever returns empty
    CONTRADICTION  = "CONTRADICTION"  # Conflicting data sources
    AMBIGUITY      = "AMBIGUITY"      # Vague task description
    RESOURCE_LIMIT = "RESOURCE_LIMIT" # Max 1 tool call allowed

print("Protocols:", [p.value for p in Protocol])
print("Failure modes:", [f.value for f in FailureMode])

Protocols: ['NL', 'JSON', 'SHARED']
Failure modes: ['NONE', 'TOOL_FAIL', 'CONTRADICTION', 'AMBIGUITY', 'RESOURCE_LIMIT']


## 2. Communication Logger
Records every message passing between agents.

In [3]:
@dataclass
class Message:
    """A single inter-agent communication event."""
    run_id:       str
    protocol:     str
    failure_mode: str
    sender:       str
    receiver:     str
    content:      Any          # Raw message content
    token_count:  int = 0
    latency_ms:   float = 0.0
    error_flag:   bool = False  # Did this message carry an error signal?
    timestamp:    float = field(default_factory=time.time)

    def token_estimate(self) -> int:
        """Rough token count: 1 token ≈ 4 chars."""
        text = json.dumps(self.content) if not isinstance(self.content, str) else self.content
        return max(1, len(text) // 4)


class CommunicationLogger:
    """Central logger — captures all inter-agent messages in a run."""

    def __init__(self):
        self.messages: List[Message] = []

    def log(self, msg: Message):
        msg.token_count = msg.token_estimate()
        self.messages.append(msg)

    def summary(self) -> Dict:
        if not self.messages:
            return {}
        total_tokens   = sum(m.token_count for m in self.messages)
        total_latency  = sum(m.latency_ms  for m in self.messages)
        error_messages = [m for m in self.messages if m.error_flag]
        return {
            "total_messages": len(self.messages),
            "total_tokens":   total_tokens,
            "total_latency_ms": round(total_latency, 2),
            "error_message_count": len(error_messages),
            "error_propagation_rate": round(len(error_messages) / len(self.messages), 3),
        }

    def clear(self):
        self.messages = []

print("✅ Logger defined")

✅ Logger defined


## 3. Mock LLM
Rule-based stub that simulates LLM responses. Replace with real API for production runs.

In [4]:
class MockLLM:
    """
    Simulates LLM behavior for each agent role.
    Returns deterministic responses based on role + input keywords.
    Inject `api_client` here to switch to a real LLM.
    """

    # Simulated base latency per role (ms)
    LATENCY = {"orchestrator": 120, "retriever": 80, "analyst": 150, "writer": 130}

    def call(self, role: str, prompt: str, protocol: Protocol) -> Tuple[str, float]:
        """Returns (response_text, latency_ms)."""
        base_latency = self.LATENCY.get(role, 100)
        # Add jitter to simulate real network variance
        latency = base_latency + random.gauss(0, 10)

        # NL is slightly slower (more tokens to parse)
        if protocol == Protocol.NL:
            latency *= 1.15

        response = self._generate(role, prompt, protocol)
        time.sleep(latency / 5000)  # Tiny sleep so timing is non-zero
        return response, round(latency, 2)

    def _generate(self, role: str, prompt: str, protocol: Protocol) -> str:
        prompt_lower = prompt.lower()

        if role == "orchestrator":
            return self._orchestrator_response(prompt_lower, protocol)
        elif role == "retriever":
            return self._retriever_response(prompt_lower, protocol)
        elif role == "analyst":
            return self._analyst_response(prompt_lower, protocol)
        elif role == "writer":
            return self._writer_response(prompt_lower, protocol)
        return "[MockLLM: unknown role]"

    def _orchestrator_response(self, prompt, protocol):
        if protocol == Protocol.JSON:
            return json.dumps({
                "task": "analyze_earnings",
                "subtasks": ["retrieve_revenue", "retrieve_eps", "analyze_trends", "write_report"],
                "target_company": "Apple",
                "fiscal_quarter": "Q1 FY2025"
            })
        return ("Please retrieve Apple Q1 FY2025 earnings data including revenue and EPS. "
                "Then analyze year-over-year trends and write a concise investment report.")

    def _retriever_response(self, prompt, protocol):
        if "empty" in prompt or "fail" in prompt:
            return "" if protocol == Protocol.NL else json.dumps({"error": "no_data", "data": None})
        if protocol == Protocol.JSON:
            return json.dumps({
                "company": "Apple Inc.",
                "quarter": "Q1 FY2025",
                "revenue_b": 124.3,
                "revenue_yoy_pct": 4.0,
                "eps": 2.40,
                "eps_yoy_pct": 10.1,
                "net_income_b": 36.3,
                "gross_margin_pct": 46.9,
                "iphone_revenue_b": 69.1,
                "services_revenue_b": 26.3
            })
        return ("Apple reported Q1 FY2025 revenue of $124.3B, up 4.0% YoY. "
                "EPS was $2.40, up 10.1% YoY. Net income reached $36.3B. "
                "Gross margin was 46.9%. iPhone revenue was $69.1B and Services revenue was $26.3B.")

    def _analyst_response(self, prompt, protocol):
        if "contradiction" in prompt or "conflict" in prompt:
            note = "WARNING: conflicting data detected in input. Analysis may be unreliable."
        elif "empty" in prompt or "no_data" in prompt or len(prompt.strip()) < 20:
            note = "ERROR: insufficient data to perform analysis."
        else:
            note = "Analysis complete."

        if protocol == Protocol.JSON:
            return json.dumps({
                "sentiment": "positive",
                "key_insights": [
                    "Services segment growing faster than hardware",
                    "EPS growth outpaces revenue growth — margin expansion",
                    "iPhone still dominant at 55.6% of revenue"
                ],
                "risk_factors": ["China market headwinds", "Saturating premium smartphone market"],
                "analyst_note": note
            })
        return (f"Apple's Q1 FY2025 results are broadly positive. "
                f"Services growth continues to outpace hardware, indicating a successful "
                f"business model transition. EPS growth of 10.1% exceeds revenue growth of 4.0%, "
                f"reflecting improved operational efficiency. Key risk: China market exposure. {note}")

    def _writer_response(self, prompt, protocol):
        if "error" in prompt.lower() or "insufficient" in prompt.lower():
            return ("[DEGRADED REPORT] Unable to generate complete analysis due to upstream data issues. "
                    "Please verify data sources and retry.")
        return ("## Apple Inc. Q1 FY2025 Earnings Summary\n\n"
                "Apple delivered solid Q1 FY2025 results, with revenue of $124.3B (+4.0% YoY) "
                "and EPS of $2.40 (+10.1% YoY). The Services segment reached $26.3B, "
                "reinforcing Apple's transition toward higher-margin recurring revenue. "
                "Gross margin of 46.9% represents a multi-year high. "
                "While iPhone revenue remains the largest segment at $69.1B, "
                "the mix shift toward Services is a positive long-term signal. "
                "Key risks include slowing China demand and premium smartphone market saturation. "
                "Overall outlook: **Positive**.")


llm = MockLLM()
print("✅ MockLLM defined")

✅ MockLLM defined


## 4. Agent Definitions

In [5]:
class BaseAgent:
    def __init__(self, name: str, role: str, llm: MockLLM, logger: CommunicationLogger):
        self.name   = name
        self.role   = role
        self.llm    = llm
        self.logger = logger

    def _send(self, receiver: str, content: Any, run_id: str,
              protocol: Protocol, failure_mode: FailureMode,
              error_flag: bool = False) -> Message:
        start = time.time()
        msg = Message(
            run_id=run_id,
            protocol=protocol.value,
            failure_mode=failure_mode.value,
            sender=self.name,
            receiver=receiver,
            content=content,
            error_flag=error_flag,
            latency_ms=round((time.time() - start) * 1000, 2)
        )
        self.logger.log(msg)
        return msg


class OrchestratorAgent(BaseAgent):
    def __init__(self, llm, logger):
        super().__init__("Orchestrator", "orchestrator", llm, logger)

    def decompose_task(self, task: str, run_id: str,
                       protocol: Protocol, failure_mode: FailureMode) -> Any:
        # Inject ambiguity failure
        if failure_mode == FailureMode.AMBIGUITY:
            task = "Look into the company's recent performance and write something about it."

        prompt = f"Decompose this task: {task}"
        response, latency = self.llm.call(self.role, prompt, protocol)

        msg = self._send("Retriever", response, run_id, protocol, failure_mode)
        msg.latency_ms = latency
        return response


class RetrieverAgent(BaseAgent):
    def __init__(self, llm, logger):
        super().__init__("Retriever", "retriever", llm, logger)
        self.tool_calls = 0
        self.max_tool_calls = 999

    def retrieve(self, task_msg: Any, run_id: str,
                 protocol: Protocol, failure_mode: FailureMode) -> Any:
        self.tool_calls += 1

        # Inject tool failure
        if failure_mode == FailureMode.TOOL_FAIL:
            prompt = "retrieve data fail empty"
        elif failure_mode == FailureMode.RESOURCE_LIMIT and self.tool_calls > self.max_tool_calls:
            prompt = "retrieve data fail empty resource limit"
        else:
            prompt = f"Retrieve earnings data for: {task_msg}"

        response, latency = self.llm.call(self.role, prompt, protocol)

        # Inject contradiction: override one field with wrong value
        if failure_mode == FailureMode.CONTRADICTION and protocol == Protocol.JSON:
            try:
                data = json.loads(response)
                data["revenue_b"] = 98.5   # Conflicting value
                data["_conflict"] = True
                response = json.dumps(data)
            except Exception:
                response += " [CONFLICT: another source reports revenue of $98.5B]"
        elif failure_mode == FailureMode.CONTRADICTION and protocol == Protocol.NL:
            response += " However, a second source reports revenue of only $98.5B, contradicting the above."

        is_error = (not response or response.strip() == "" or
                    (protocol == Protocol.JSON and "no_data" in response))

        msg = self._send("Analyst", response, run_id, protocol, failure_mode, error_flag=is_error)
        msg.latency_ms = latency
        return response


class AnalystAgent(BaseAgent):
    def __init__(self, llm, logger):
        super().__init__("Analyst", "analyst", llm, logger)

    def analyze(self, retrieved_data: Any, run_id: str,
                protocol: Protocol, failure_mode: FailureMode) -> Any:
        # Detect upstream error
        data_str = json.dumps(retrieved_data) if not isinstance(retrieved_data, str) else retrieved_data
        prompt = f"Analyze this financial data: {data_str}"

        response, latency = self.llm.call(self.role, prompt, protocol)

        # Did the analyst detect the error?
        detected = ("warning" in response.lower() or
                    "error" in response.lower() or
                    "insufficient" in response.lower() or
                    "conflict" in response.lower())

        # Propagate error flag if not detected
        upstream_error = (not retrieved_data or
                          (isinstance(retrieved_data, str) and len(retrieved_data.strip()) < 10))
        propagate_error = upstream_error and not detected

        msg = self._send("Writer", response, run_id, protocol, failure_mode,
                         error_flag=propagate_error)
        msg.latency_ms = latency
        msg.token_count = msg.token_estimate()

        return response, detected


class WriterAgent(BaseAgent):
    def __init__(self, llm, logger):
        super().__init__("Writer", "writer", llm, logger)

    def write_report(self, analysis: Any, run_id: str,
                     protocol: Protocol, failure_mode: FailureMode) -> str:
        data_str = json.dumps(analysis) if not isinstance(analysis, str) else analysis
        prompt = f"Write a report based on this analysis: {data_str}"
        response, latency = self.llm.call(self.role, prompt, protocol)

        is_degraded = "DEGRADED" in response or "unable" in response.lower()
        msg = self._send("Output", response, run_id, protocol, failure_mode,
                         error_flag=is_degraded)
        msg.latency_ms = latency
        return response


print("✅ All agents defined")

✅ All agents defined


## 5. Shared Memory Protocol
All agents read/write a single global state dict — no explicit message passing.

In [6]:
class SharedMemory:
    """Global state accessible by all agents."""
    def __init__(self):
        self._state: Dict = {}
        self._write_log: List[Dict] = []  # Track who wrote what

    def write(self, agent: str, key: str, value: Any):
        self._state[key] = value
        self._write_log.append({"agent": agent, "key": key, "ts": time.time()})

    def read(self, key: str, default=None) -> Any:
        return self._state.get(key, default)

    def read_all(self) -> Dict:
        return copy.deepcopy(self._state)

    def token_footprint(self) -> int:
        """Total tokens in the shared state at this moment."""
        return max(1, len(json.dumps(self._state)) // 4)

    def clear(self):
        self._state = {}
        self._write_log = []

print("✅ SharedMemory defined")

✅ SharedMemory defined


## 6. Pipeline Runner
Orchestrates a full run for a given (protocol, failure_mode) combination.

In [7]:
@dataclass
class RunResult:
    run_id:              str
    protocol:            str
    failure_mode:        str
    total_tokens:        int
    total_latency_ms:    float
    total_messages:      int
    error_count:         int
    error_propagation_rate: float
    error_detected:      bool   # Did Analyst catch the upstream error?
    report_degraded:     bool   # Did the final report show degradation?
    report_snippet:      str    # First 120 chars of final report
    recovery_success:    bool   # True if error detected AND report not degraded


def run_pipeline(protocol: Protocol,
                 failure_mode: FailureMode,
                 task: str = "Analyze Apple Q1 FY2025 earnings report",
                 resource_limit: int = 999,
                 seed: int = 42) -> RunResult:
    random.seed(seed)
    run_id = hashlib.md5(f"{protocol}{failure_mode}{seed}".encode()).hexdigest()[:8]

    logger = CommunicationLogger()
    shared_mem = SharedMemory()

    orch      = OrchestratorAgent(llm, logger)
    retriever = RetrieverAgent(llm, logger)
    analyst   = AnalystAgent(llm, logger)
    writer    = WriterAgent(llm, logger)
    retriever.max_tool_calls = resource_limit

    # ── SHARED MEMORY PROTOCOL ──────────────────────────────────────────────
    if protocol == Protocol.SHARED:
        # Orchestrator writes task to shared mem
        task_content = orch.decompose_task(task, run_id, protocol, failure_mode)
        shared_mem.write("Orchestrator", "task", task_content)

        # Retriever reads task, writes data
        raw_task = shared_mem.read("task", "")
        retrieved = retriever.retrieve(raw_task, run_id, protocol, failure_mode)
        shared_mem.write("Retriever", "retrieved_data", retrieved)

        # Analyst reads data, writes analysis
        raw_data = shared_mem.read("retrieved_data", "")
        analysis, error_detected = analyst.analyze(raw_data, run_id, protocol, failure_mode)
        shared_mem.write("Analyst", "analysis", analysis)

        # Writer reads analysis, writes report
        raw_analysis = shared_mem.read("analysis", "")
        report = writer.write_report(raw_analysis, run_id, protocol, failure_mode)
        shared_mem.write("Writer", "report", report)

        # Add shared memory token footprint to total
        extra_tokens = shared_mem.token_footprint()

    # ── NL and JSON PROTOCOLS ───────────────────────────────────────────────
    else:
        task_msg   = orch.decompose_task(task, run_id, protocol, failure_mode)
        retrieved  = retriever.retrieve(task_msg, run_id, protocol, failure_mode)
        analysis, error_detected = analyst.analyze(retrieved, run_id, protocol, failure_mode)
        report     = writer.write_report(analysis, run_id, protocol, failure_mode)
        extra_tokens = 0

    # ── COLLECT METRICS ─────────────────────────────────────────────────────
    summary = logger.summary()
    report_degraded = "DEGRADED" in report or "unable" in report.lower()
    recovery_success = error_detected and not report_degraded

    return RunResult(
        run_id=run_id,
        protocol=protocol.value,
        failure_mode=failure_mode.value,
        total_tokens=summary.get("total_tokens", 0) + extra_tokens,
        total_latency_ms=summary.get("total_latency_ms", 0),
        total_messages=summary.get("total_messages", 0),
        error_count=summary.get("error_message_count", 0),
        error_propagation_rate=summary.get("error_propagation_rate", 0.0),
        error_detected=error_detected,
        report_degraded=report_degraded,
        report_snippet=report[:120],
        recovery_success=recovery_success,
    )

print("✅ Pipeline runner defined")

✅ Pipeline runner defined


## 7. Full Experiment Grid
3 protocols × 5 failure modes × N repetitions

In [8]:
N_REPS = 20  # Repetitions per cell — increase for tighter CIs in real study

results = []

for protocol in Protocol:
    for failure_mode in FailureMode:
        for rep in range(N_REPS):
            rl = 1 if failure_mode == FailureMode.RESOURCE_LIMIT else 999
            r = run_pipeline(protocol, failure_mode,
                             resource_limit=rl, seed=rep)
            results.append(asdict(r))

df = pd.DataFrame(results)
print(f"✅ Experiment complete: {len(df)} runs ({len(Protocol)} protocols × "
      f"{len(FailureMode)} failure modes × {N_REPS} reps)")
df.head()

✅ Experiment complete: 300 runs (3 protocols × 5 failure modes × 20 reps)


,run_id,protocol,failure_mode,total_tokens,total_latency_ms,total_messages,error_count,error_propagation_rate,error_detected,report_degraded,report_snippet,recovery_success
0,8bacfa3c,NL,NONE,293,543.21,4,0,0.0,False,False,## Apple Inc. Q1 FY2025 Earnings Summary\n\nAp...,False
1,ab330206,NL,NONE,293,575.45,4,0,0.0,False,False,## Apple Inc. Q1 FY2025 Earnings Summary\n\nAp...,False
2,73f387d2,NL,NONE,293,577.49,4,0,0.0,False,False,## Apple Inc. Q1 FY2025 Earnings Summary\n\nAp...,False
3,b127a801,NL,NONE,293,568.17,4,0,0.0,False,False,## Apple Inc. Q1 FY2025 Earnings Summary\n\nAp...,False
4,40cedafd,NL,NONE,293,556.58,4,0,0.0,False,False,## Apple Inc. Q1 FY2025 Earnings Summary\n\nAp...,False


## 8. Research Question 1 — Protocol Efficiency (Baseline)
Under normal operation (NONE), which protocol uses fewer tokens and is faster?

In [9]:
baseline = df[df["failure_mode"] == "NONE"]

eff = baseline.groupby("protocol").agg(
    mean_tokens   = ("total_tokens",      "mean"),
    std_tokens    = ("total_tokens",       "std"),
    mean_latency  = ("total_latency_ms",  "mean"),
    std_latency   = ("total_latency_ms",   "std"),
    mean_messages = ("total_messages",    "mean"),
).round(2)

print("=" * 60)
print("RQ1: Protocol Efficiency (Baseline — no failures)")
print("=" * 60)
print(eff.to_string())

# One-way ANOVA on token usage across protocols
groups = [baseline[baseline["protocol"] == p]["total_tokens"].values for p in ["NL", "JSON", "SHARED"]]
f_stat, p_val = stats.f_oneway(*groups)
print(f"\nOne-way ANOVA (token usage): F={f_stat:.3f}, p={p_val:.4f}")
print("→ Significant difference" if p_val < 0.05 else "→ No significant difference")

RQ1: Protocol Efficiency (Baseline — no failures)
          mean_tokens  std_tokens  mean_latency  std_latency  mean_messages
protocol                                                                   
JSON            314.0         0.0        477.82        22.24            4.0
NL              293.0         0.0        549.49        25.57            4.0
SHARED          604.0         0.0        477.82        22.24            4.0

One-way ANOVA (token usage): F=inf, p=0.0000
→ Significant difference


## 9. Research Question 2 — Failure Propagation
Under stress, which protocol best detects and contains errors?

In [10]:
failures = df[df["failure_mode"] != "NONE"]

fail_summary = failures.groupby(["protocol", "failure_mode"]).agg(
    error_detection_rate  = ("error_detected",         "mean"),
    error_propagation_rate = ("error_propagation_rate", "mean"),
    recovery_success_rate = ("recovery_success",        "mean"),
    report_degradation_rate = ("report_degraded",       "mean"),
).round(3)

print("=" * 70)
print("RQ2: Failure Propagation by Protocol × Failure Mode")
print("=" * 70)
print(fail_summary.to_string())

# Two-way ANOVA: protocol × failure_mode on propagation rate
from itertools import product
import statsmodels.api as sm
from statsmodels.formula.api import ols

try:
    model = ols("error_propagation_rate ~ C(protocol) + C(failure_mode) + C(protocol):C(failure_mode)",
                data=failures).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print("\nTwo-way ANOVA (error propagation rate):")
    print(anova_table.round(4))
except ImportError:
    print("\n[statsmodels not available — run: pip install statsmodels]")

RQ2: Failure Propagation by Protocol × Failure Mode
                         error_detection_rate  error_propagation_rate  recovery_success_rate  report_degradation_rate
protocol failure_mode                                                                                                
JSON     AMBIGUITY                        0.0                    0.00                    0.0                      0.0
         CONTRADICTION                    1.0                    0.00                    1.0                      0.0
         RESOURCE_LIMIT                   0.0                    0.00                    0.0                      0.0
         TOOL_FAIL                        1.0                    0.50                    0.0                      1.0
NL       AMBIGUITY                        0.0                    0.00                    0.0                      0.0
         CONTRADICTION                    0.0                    0.00                    0.0                      0.0
    

## 10. Bootstrap Confidence Intervals

In [11]:
def bootstrap_ci(data: np.ndarray, stat_fn=np.mean, n_boot=2000, alpha=0.05) -> Tuple[float, float]:
    boot_stats = [stat_fn(np.random.choice(data, size=len(data), replace=True)) for _ in range(n_boot)]
    lo = np.percentile(boot_stats, 100 * alpha / 2)
    hi = np.percentile(boot_stats, 100 * (1 - alpha / 2))
    return round(lo, 3), round(hi, 3)

print("Bootstrap 95% CIs — Token Usage (Baseline)")
print("-" * 45)
for protocol in ["NL", "JSON", "SHARED"]:
    data = baseline[baseline["protocol"] == protocol]["total_tokens"].values
    lo, hi = bootstrap_ci(data)
    mean   = round(np.mean(data), 1)
    print(f"  {protocol:<8}  mean={mean:6.1f}   95% CI [{lo}, {hi}]")

print("\nBootstrap 95% CIs — Error Detection Rate (all failure modes)")
print("-" * 55)
for protocol in ["NL", "JSON", "SHARED"]:
    data = failures[failures["protocol"] == protocol]["error_detected"].values.astype(float)
    lo, hi = bootstrap_ci(data)
    mean   = round(np.mean(data), 3)
    print(f"  {protocol:<8}  mean={mean:.3f}   95% CI [{lo}, {hi}]")

Bootstrap 95% CIs — Token Usage (Baseline)
---------------------------------------------
  NL        mean= 293.0   95% CI [293.0, 293.0]
  JSON      mean= 314.0   95% CI [314.0, 314.0]
  SHARED    mean= 604.0   95% CI [604.0, 604.0]

Bootstrap 95% CIs — Error Detection Rate (all failure modes)
-------------------------------------------------------
  NL        mean=0.000   95% CI [0.0, 0.0]
  JSON      mean=0.500   95% CI [0.4, 0.612]
  SHARED    mean=0.250   95% CI [0.162, 0.35]


## 11. Markov Chain — Error Propagation Model
States: `OK` → `ERROR_INJECTED` → `ERROR_DETECTED` / `ERROR_PROPAGATED`

In [12]:
def build_transition_matrix(protocol_df: pd.DataFrame) -> np.ndarray:
    """
    States:
      0 = OK (no error injected)
      1 = ERROR_INJECTED (Retriever sends bad data)
      2 = ERROR_DETECTED (Analyst catches it)
      3 = ERROR_PROPAGATED (error passes through to Writer)

    Transition counts estimated from run data.
    """
    n_runs   = len(protocol_df)
    no_fail  = (protocol_df["failure_mode"] == "NONE").sum()
    has_fail = n_runs - no_fail

    p_inject  = has_fail / n_runs if n_runs > 0 else 0
    p_detect  = protocol_df["error_detected"].mean()
    p_recover = protocol_df["recovery_success"].mean()

    # Row = from state, Col = to state
    T = np.array([
        [1 - p_inject, p_inject,       0,                 0],           # OK
        [0,            0,               p_detect,     1 - p_detect],    # ERROR_INJECTED
        [p_recover,    1 - p_recover,   0,                 0],          # ERROR_DETECTED
        [0,            0,               0,                 1],          # ERROR_PROPAGATED (absorbing)
    ])
    return T

print("Markov Transition Matrices — Error Propagation")
print("=" * 55)
states = ["OK", "ERR_INJECT", "ERR_DETECT", "ERR_PROP"]

for protocol in ["NL", "JSON", "SHARED"]:
    pdata = df[df["protocol"] == protocol]
    T = build_transition_matrix(pdata)
    print(f"\n[{protocol}]")
    header = f"{'':>12}" + "".join(f"{s:>13}" for s in states)
    print(header)
    for i, row_label in enumerate(states):
        row_str = f"{row_label:>12}" + "".join(f"{T[i,j]:>13.3f}" for j in range(4))
        print(row_str)

    # Steady-state absorption probability into ERROR_PROPAGATED
    # Run 10-step simulation from state 0
    state_vec = np.array([1, 0, 0, 0], dtype=float)
    for _ in range(10):
        state_vec = state_vec @ T
    print(f"  → P(absorbed into ERROR_PROPAGATED after 10 steps): {state_vec[3]:.3f}")

Markov Transition Matrices — Error Propagation

[NL]
                       OK   ERR_INJECT   ERR_DETECT     ERR_PROP
          OK        0.200        0.800        0.000        0.000
  ERR_INJECT        0.000        0.000        0.000        1.000
  ERR_DETECT        0.000        1.000        0.000        0.000
    ERR_PROP        0.000        0.000        0.000        1.000
  → P(absorbed into ERROR_PROPAGATED after 10 steps): 1.000

[JSON]
                       OK   ERR_INJECT   ERR_DETECT     ERR_PROP
          OK        0.200        0.800        0.000        0.000
  ERR_INJECT        0.000        0.000        0.400        0.600
  ERR_DETECT        0.200        0.800        0.000        0.000
    ERR_PROP        0.000        0.000        0.000        1.000
  → P(absorbed into ERROR_PROPAGATED after 10 steps): 0.975

[SHARED]
                       OK   ERR_INJECT   ERR_DETECT     ERR_PROP
          OK        0.200        0.800        0.000        0.000
  ERR_INJECT        0.000    

## 12. Summary Table — Protocol Comparison

In [13]:
summary_rows = []

for protocol in ["NL", "JSON", "SHARED"]:
    base_p = df[(df["protocol"] == protocol) & (df["failure_mode"] == "NONE")]
    fail_p = df[(df["protocol"] == protocol) & (df["failure_mode"] != "NONE")]

    summary_rows.append({
        "Protocol":              protocol,
        "Avg Tokens (baseline)": round(base_p["total_tokens"].mean(), 1),
        "Avg Latency ms (base)": round(base_p["total_latency_ms"].mean(), 1),
        "Error Detection Rate":  round(fail_p["error_detected"].mean(), 3),
        "Error Propagation Rate": round(fail_p["error_propagation_rate"].mean(), 3),
        "Recovery Success Rate": round(fail_p["recovery_success"].mean(), 3),
        "Report Degradation Rate": round(fail_p["report_degraded"].mean(), 3),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Protocol")

print("=" * 80)
print("FINAL PROTOCOL COMPARISON SUMMARY")
print("=" * 80)
print(summary_df.to_string())
print()
print("Interpretation guide:")
print("  Tokens/Latency  → lower is better (efficiency)")
print("  Error Detection → higher is better (reliability)")
print("  Error Propagation → lower is better (containment)")
print("  Recovery Success  → higher is better (resilience)")

FINAL PROTOCOL COMPARISON SUMMARY
          Avg Tokens (baseline)  Avg Latency ms (base)  Error Detection Rate  Error Propagation Rate  Recovery Success Rate  Report Degradation Rate
Protocol                                                                                                                                            
NL                        293.0                  549.5                  0.00                   0.125                   0.00                     0.00
JSON                      314.0                  477.8                  0.50                   0.125                   0.25                     0.25
SHARED                    604.0                  477.8                  0.25                   0.062                   0.00                     0.25

Interpretation guide:
  Tokens/Latency  → lower is better (efficiency)
  Error Detection → higher is better (reliability)
  Error Propagation → lower is better (containment)
  Recovery Success  → higher is better (resili

## 13. Next Steps — Real LLM Integration

To run this with a real LLM (DeepSeek / GPT-4o-mini), replace `MockLLM.call()` with:

```python
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_KEY",
    base_url="https://api.deepseek.com"  # DeepSeek-compatible endpoint
)

def real_llm_call(role: str, prompt: str, system_prompt: str) -> Tuple[str, float, int]:
    start = time.time()
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": f"You are a {role} agent in a multi-agent pipeline."},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.3
    )
    latency = (time.time() - start) * 1000
    text    = resp.choices[0].message.content
    tokens  = resp.usage.total_tokens
    return text, latency, tokens
```

Also:
- Replace mock data in `RetrieverAgent` with real `akshare` or web search calls
- Increase `N_REPS` to 50+ for statistical power
- Add a proper task benchmark set (20+ diverse questions)